# ILSA Fine-tuning v2 — Colab A100 Pipeline

**Önce:** Runtime → Change runtime type → **A100**  
**Sonra:** Hücreleri sırayla çalıştır (Shift+Enter veya Runtime → Run all)

Adımlar:
1. Ortam kurulumu (repo clone + paketler)
2. Memory probe (VRAM kontrolü)
3. LoRA eğitimi (~2-3 saat)
4. Her epoch için extraction (checkpoint seçimi için)
5. En iyi checkpoint seçimi
6. A0 (base) ve A1 (LoRA) extraction
7. A0 ve A1 eval
8. Sonuçları Drive'a kaydet

## 0. Runtime kontrolü

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())
assert 'A100' in result.stdout, 'HATA: A100 runtime seçili değil! Runtime → Change runtime type → A100 seç.'

## 1. Repo ve paketler

In [ ]:
# GitHub token gir (private repo ise)
# Public ise TOKEN satırını sil
import os
TOKEN = ''  # <-- buraya GitHub Personal Access Token yaz (repo:read yetkisi yeterli)
            # Almak için: github.com → Settings → Developer settings → Personal access tokens

BRANCH = 'feat/finetuning-v2-pipeline'
REPO   = 'dedemerve/ILSA-LLM-Extractor'

if TOKEN:
    clone_url = f'https://{TOKEN}@github.com/{REPO}.git'
else:
    clone_url = f'https://github.com/{REPO}.git'

!git clone --branch {BRANCH} --depth 1 {clone_url} /content/ILSA_LLMs
%cd /content/ILSA_LLMs

In [ ]:
!pip install -r requirements-finetuning-v2.txt -q

## 2. Memory probe (VRAM yeterli mi?)

In [ ]:
!python3 scripts/finetuning/train_lora_v2.py \
    --model Qwen/Qwen3-8B \
    --out outputs/finetuning/v2/runs/qwen3_8b_r16 \
    --epochs 3 --r 16 --batch-size 1 \
    --memory-probe

# Beklenen: {"max_tokens": ..., "peak_gb": ...}
# peak_gb < 38 olmalı. Değilse aşağıdaki hücreyi çalıştır.

In [ ]:
# SADECE peak_gb >= 38 ise çalıştır:
# !python3 scripts/finetuning/train_lora_v2.py \
#     --model Qwen/Qwen3-8B \
#     --out outputs/finetuning/v2/runs/qwen3_8b_r8 \
#     --epochs 3 --r 8 --batch-size 1 --grad-accum 8 \
#     --memory-probe

## 3. LoRA eğitimi (~2-3 saat)

Gate assertions otomatik çalışır. Herhangi biri FAIL derse dur, Merve'ye bildir.

In [ ]:
!python3 scripts/finetuning/train_lora_v2.py \
    --model Qwen/Qwen3-8B \
    --out outputs/finetuning/v2/runs/qwen3_8b_r16 \
    --epochs 3 --r 16 --batch-size 1 --grad-accum 4

# Beklenen çıktı satırları:
# gate: {"n": 107, "max_tokens": ..., "supervised_tail_example": ...}
# run_manifest.json içinde: "adapter_reload_verified": true

In [ ]:
# Eğitim bitti mi ve kaç checkpoint var?
import json
from pathlib import Path

manifest_path = Path('outputs/finetuning/v2/runs/qwen3_8b_r16/run_manifest.json')
assert manifest_path.exists(), 'run_manifest.json yok — eğitim tamamlanmadı!'
m = json.loads(manifest_path.read_text())
print('Checkpoints:', m['epoch_checkpoints'])
print('Adapter reload verified:', m['adapter_reload_verified'])
assert m['adapter_reload_verified'], 'Adapter reload FAILED!'

CHECKPOINTS = [Path(p).name for p in m['epoch_checkpoints']]  # örn: ['checkpoint-27', ...]
print('\nKullanılacak checkpoint adları:', CHECKPOINTS)

## 4. Her epoch checkpoint için extraction (select_checkpoint için)

In [ ]:
# Her checkpoint için extract_v2.py çalıştırır
# Arm adı: qwen3_8b_r16_<checkpoint-adı>  (select_checkpoint bunu arar)

for ckpt in CHECKPOINTS:
    adapter_path = f'outputs/finetuning/v2/runs/qwen3_8b_r16/{ckpt}'
    arm = f'qwen3_8b_r16_{ckpt}'
    print(f'\n=== {arm} ===')
    !python3 scripts/finetuning/extract_v2.py \
        --mode chunked --backend vllm \
        --model Qwen/Qwen3-8B \
        --adapter {adapter_path} \
        --split val --arm {arm}

## 5. En iyi checkpoint seç

In [ ]:
!python3 scripts/finetuning/select_checkpoint.py --split val

# Sonuç: outputs/finetuning/v2/eval/checkpoint_selection_val.json
sel = json.loads(Path('outputs/finetuning/v2/eval/checkpoint_selection_val.json').read_text())
BEST_ADAPTER = sel['selected']['adapter']
print('\nSEÇİLEN ADAPTER:', BEST_ADAPTER)
print('Strict F1:', sel['selected']['strict_f1'])

## 6. A0 (base) ve A1 (LoRA) extraction

In [ ]:
# A0: adapter yok
!python3 scripts/finetuning/extract_v2.py \
    --mode chunked --backend vllm \
    --model Qwen/Qwen3-8B \
    --split val --arm A0_base_qwen3_chunked

In [ ]:
# A1: en iyi LoRA checkpoint
!python3 scripts/finetuning/extract_v2.py \
    --mode chunked --backend vllm \
    --model Qwen/Qwen3-8B \
    --adapter {BEST_ADAPTER} \
    --split val --arm A1_qwen3_lora_selected

## 7. A0 ve A1 eval

In [ ]:
!python3 scripts/finetuning/eval_extraction.py \
    --pred outputs/finetuning/v2/predictions/A0_base_qwen3_chunked_val.json \
    --split val --label A0 \
    --out outputs/finetuning/v2/eval/A0_base_qwen3_val.json

a0 = json.loads(Path('outputs/finetuning/v2/eval/A0_base_qwen3_val.json').read_text())
a0_s = a0['A0']['summary']
print(f"A0  strict={a0_s['strict']['f1']}  lenient={a0_s['lenient']['f1']}  legacy={a0_s['legacy_greedy_construct']['f1']}")

In [ ]:
!python3 scripts/finetuning/eval_extraction.py \
    --pred outputs/finetuning/v2/predictions/A1_qwen3_lora_selected_val.json \
    --split val --label A1 \
    --out outputs/finetuning/v2/eval/A1_qwen3_lora_val.json

a1 = json.loads(Path('outputs/finetuning/v2/eval/A1_qwen3_lora_val.json').read_text())
a1_s = a1['A1']['summary']
print(f"A1  strict={a1_s['strict']['f1']}  lenient={a1_s['lenient']['f1']}  legacy={a1_s['legacy_greedy_construct']['f1']}")

print()
if a1_s['strict']['f1'] > a0_s['strict']['f1']:
    print('H1 DESTEKLENDI: A1 strict F1 > A0 strict F1')
else:
    print('H1 desteklenmedi (A1 <= A0 strict F1) — raporlanabilir, başarısızlık değil')

## 8. Çıktıları Drive'a kaydet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DEST = '/content/drive/MyDrive/ILSA_finetuning_v2_outputs'
shutil.copytree('outputs/finetuning/v2', DEST, dirs_exist_ok=True)
print('Kaydedildi:', DEST)

In [ ]:
# Özet — bu değerleri Merve'ye ilet
print('=== SONUÇ ÖZETİ ===')
print(f"A0  strict={a0_s['strict']['f1']}  lenient={a0_s['lenient']['f1']}  legacy={a0_s['legacy_greedy_construct']['f1']}")
print(f"A1  strict={a1_s['strict']['f1']}  lenient={a1_s['lenient']['f1']}  legacy={a1_s['legacy_greedy_construct']['f1']}")
print(f"Seçilen adapter: {BEST_ADAPTER}")

## 9. Tam korpus extraction — A1_fullcorpus_qwen3_lora (1266 çalışma, ~7-9 saat)

Sadece A1 checkpoint-81 adaptörü ile çalışır. Çıktı: `outputs/finetuning/v2/predictions/A1_fullcorpus_qwen3_lora_corpus.json`

**Ön koşul:** Adım 5'te `BEST_ADAPTER` seçilmiş olmalı (checkpoint-81).

In [ ]:
# Corpus run başlatmadan önce kontrol: kaç çalışmada metin var?
import pandas as pd
inv = pd.read_csv('outputs/finetuning/source_recovery/full_source_inventory.csv', dtype=str)
n_corpus = (inv['text_extracted'].str.strip().str.lower() == 'true').sum()
print(f'Metin çıkarılmış çalışma sayısı: {n_corpus}')
print(f'Seçilen adapter: {BEST_ADAPTER}')
print()
print('Devam etmek için bir sonraki hücreyi çalıştır.')

In [ ]:
# Tam korpus extraction (~7-9 saat, A100 gerekli)
# ARM: A1_fullcorpus_qwen3_lora
# Çıktı: outputs/finetuning/v2/predictions/A1_fullcorpus_qwen3_lora_corpus.json

!python3 scripts/finetuning/extract_v2.py \
    --mode chunked --backend vllm \
    --model Qwen/Qwen3-8B \
    --adapter {BEST_ADAPTER} \
    --split corpus \
    --arm A1_fullcorpus_qwen3_lora

print('\nKorpus extraction tamamlandı.')
print('Çıktı dosyası: outputs/finetuning/v2/predictions/A1_fullcorpus_qwen3_lora_corpus.json')

In [ ]:
# Sonucu Drive'a kaydet
DEST_CORPUS = '/content/drive/MyDrive/ILSA_finetuning_v2_outputs'
import shutil, json
shutil.copy(
    'outputs/finetuning/v2/predictions/A1_fullcorpus_qwen3_lora_corpus.json',
    f'{DEST_CORPUS}/predictions/A1_fullcorpus_qwen3_lora_corpus.json'
)

# Kısa istatistik
preds = json.loads(open('outputs/finetuning/v2/predictions/A1_fullcorpus_qwen3_lora_corpus.json').read())
n_studies = len(preds['predictions'])
n_ok = sum(p['status'] == 'ok' for p in preds['predictions'])
n_missing = sum(p['status'] == 'missing_source' for p in preds['predictions'])
total_pairs = sum(len(p['pairs']) for p in preds['predictions'])
print(f'Çalışma sayısı   : {n_studies}')
print(f'ok               : {n_ok}')
print(f'missing_source   : {n_missing}')
print(f'Toplam çift      : {total_pairs}')
print(f'Drive kayıt      : {DEST_CORPUS}/predictions/')